# V-MD3 SAR Data Acquisition

Headless continuous-motion SAR data collection for the V-MD3 radar with a Galil-driven CDS-3310 stage. Each sweep records RADC / RFFT / DONE frames to a `.bin` file while the stage moves, with the stage position logged for every frame.

**One-time install (in your venv):** `py -m pip install https://www.galil.com/sw/pub/python/gclib-1.0.0-py3-none-any.whl`

**Before running:** close GDK, connect the radar and the Galil, then run Cell 0 (approve the UAC prompt). Use `python measure.py` separately for the live heatmap.

**Workflow:**

- **Cell 0** - Radar network setup (re-run after replugging the radar).
- **Cell 1** - Imports and helper functions (run once).
- **Cell 2** - User configuration.
- **Cell 3** - Connect to the radar.
- **Cell 3B** - Connect to the Galil and home the stage.
- **Cell 3C** - Measure the frame period and sweep speed.
- **Cell 4** - One continuous sweep (repeat per sweep).
- **Cell 5** - Disconnect radar and Galil.

## Cell 0 - Radar network setup

Runs `setup_radar.ps1` (same folder as this notebook) as administrator. A UAC prompt appears unless Jupyter is already elevated. The script output prints here when it finishes.

In [ ]:
import os
import base64
import ctypes
import subprocess
from pathlib import Path

# Setup script location and a temp log the elevated process writes to
SETUP_PS1 = Path("setup_radar.ps1").resolve()
SETUP_LOG = Path(os.environ["TEMP"]) / "vmd3_setup_log.txt"


def run_radar_setup(ps1=SETUP_PS1, log=SETUP_LOG):
    """Run setup_radar.ps1 as administrator and print its output here."""
    if not ps1.is_file():
        raise FileNotFoundError(f"Setup script not found: {ps1}")
    log.unlink(missing_ok=True)

    # Build a PowerShell command that runs the script and saves all output to the log
    ps1_q = str(ps1).replace("'", "''")
    log_q = str(log).replace("'", "''")
    inner = f"& '{ps1_q}' *>&1 | Out-File -FilePath '{log_q}' -Encoding utf8"
    encoded = base64.b64encode(inner.encode("utf-16-le")).decode()
    ps_args = ["-NoProfile", "-ExecutionPolicy", "Bypass", "-EncodedCommand", encoded]

    # Run directly if already elevated, otherwise go through a UAC prompt
    if ctypes.windll.shell32.IsUserAnAdmin():
        subprocess.run(["powershell", *ps_args])
    else:
        print("[cell 0] Approve the UAC prompt to run the radar setup...")
        arg_str = " ".join(ps_args)
        launcher = f"Start-Process powershell -Verb RunAs -Wait -WindowStyle Hidden -ArgumentList '{arg_str}'"
        subprocess.run(["powershell", "-NoProfile", "-Command", launcher])

    # Show the script output and confirm it passed
    if not log.is_file():
        raise RuntimeError("No setup output. The UAC prompt may have been declined.")
    text = log.read_text(encoding="utf-8-sig")
    print(text)
    if "All checks passed" not in text:
        raise RuntimeError("Radar setup did not pass. See the messages above.")
    print("[cell 0] Radar network ready.")


run_radar_setup()

## Cell 1 - Imports and helper functions

In [ ]:
import os
import csv
import time
import socket
import numpy as np
import pandas as pd
import gclib
from datetime import datetime
from lib.vmd3 import VMD3, RdotConfig

# Map user-facing stream names to RdotConfig flags and UDP headers
STREAM_MAP = {
    'radc': RdotConfig.RADC,
    'rfft': RdotConfig.RFFT,
    'done': RdotConfig.DONE,
}
HEADER_MAP = {
    'radc': b'RADC',
    'rfft': b'RFFT',
    'done': b'DONE',
}
CYCLE_ORDER = ["radc", "rfft", "done"]   # order the streams arrive within one radar cycle


def read_any_frame(vmd3, headers):
    """Read the next UDP frame whose header is in `headers`. Returns full frame."""
    while True:
        data = vmd3.recv_udp()
        if data[0:4] not in headers:
            continue
        resp_len = int.from_bytes(data[4:8], byteorder='little')
        while len(data) < resp_len + 8:
            data += vmd3.recv_udp()
        if resp_len != len(data[8:]):
            continue
        return data


def drain_udp(vmd3):
    """Throw away everything already sitting in the UDP buffer."""
    old_timeout = vmd3.sockUDP.gettimeout()
    vmd3.sockUDP.settimeout(0.05)
    try:
        while True:
            try:
                vmd3.recv_udp()
            except socket.timeout:
                break
    finally:
        vmd3.sockUDP.settimeout(old_timeout)


def wait_for_frame(vmd3, header):
    """Block until the next complete frame with this header arrives."""
    while True:
        try:
            return read_any_frame(vmd3, (header,))
        except socket.timeout:
            continue


def sync_stream():
    """Last selected stream in each radar cycle, used to mark cycle boundaries."""
    return [s for s in CYCLE_ORDER if s in STREAMS][-1]


def gcmd(cmd):
    """Send one Galil command and return its stripped response."""
    try:
        return galil.GCommand(cmd).strip()
    except gclib.GclibError as e:
        raise RuntimeError(f"Galil rejected '{cmd}': {galil.GCommand('TC1').strip()}") from e


def galil_state():
    """Return (position in counts, True if the axis is still moving)."""
    pos = float(gcmd(f"MG _TP{GALIL_AXIS}"))
    moving = float(gcmd(f"MG _BG{GALIL_AXIS}")) > 0.5
    return int(pos), moving


def galil_move(counts, speed, absolute=False):
    """Move the stage at a given speed and block until the move finishes."""
    gcmd(f"SP{GALIL_AXIS}={int(speed)}")
    gcmd(f"{'PA' if absolute else 'PR'}{GALIL_AXIS}={int(counts)}")
    gcmd(f"BG{GALIL_AXIS}")
    galil.GMotionComplete(GALIL_AXIS)


def galil_home():
    """Drive into the start end stop, back off, and call that position zero."""
    gcmd(f"SH{GALIL_AXIS}")
    gcmd(f"AC{GALIL_AXIS}={SWEEP_ACCEL}")
    gcmd(f"DC{GALIL_AXIS}={SWEEP_ACCEL}")
    galil_move(START_END_SIGN * HOME_OVERTRAVEL, HOME_SPEED)
    gcmd(f"SH{GALIL_AXIS}")   # re-enable in case the end stop tripped the error limit
    galil_move(-START_END_SIGN * HOME_BACKOFF, HOME_SPEED)
    time.sleep(0.2)
    gcmd(f"DP{GALIL_AXIS}=0")


def galil_go_to_start():
    """Return to the zero position set by galil_home and let it settle."""
    galil_move(0, HOME_SPEED, absolute=True)
    time.sleep(0.2)


print('[cell 1] Imports and helpers loaded.')

## Cell 2 - Configuration

Edit values below for the session, then run the cell. Re-run it any time something is changed.

- **Changing `SESSION_NAME` or sweep settings** only needs this cell re-run, then go to Cell 4.
- **Changing `RSET_CONFIG` or `STREAMS`** needs a reconnect: re-run this cell, then Cell 3 and Cell 3C.

In [ ]:
# ─── Where to save binary data ───
OUTPUT_BASE = "data"   # base directory for all sessions
SESSION_NAME = "20260827_measurements"   # folder under OUTPUT_BASE

# ─── What to record ───
STREAMS = ["radc", "rfft", "done"]   # any of: "radc", "rfft", "done"

# ─── Radar mode ───
RSET_CONFIG = 0   # 0 = 2D, 6 m max range, 4.69 cm range bin

# ─── Connection (no need change) ───
RADAR_IP = "192.168.100.201"
UDP_TIMEOUT = 1.0   # seconds

# ─── Continuous sweep (Cell 4) ───
SWEEP_LENGTH_MM = 100.0
N_SWEEP_FRAMES = 101
FRAME_PERIOD_OVERRIDE_S = None   # None uses the Cell 3C measurement, or set e.g. 0.131
SWEEP_ACCEL = 1_000_000   # counts/s^2, reaches sweep speed in under 0.1 s
SWEEP_BASENAME = "sweep"

# ─── Galil stage (CDS-3310) ───
GALIL_ADDRESS = "192.168.0.100 -d"   # same address GDK connects to
GALIL_AXIS = "A"
COUNTS_PER_MM = 10_000   # 1 count = 100 nm
STAGE_RANGE_COUNTS = 1_021_243
START_END_SIGN = +1   # +1 starts at the end the old PR +1100000 drove to, -1 for the other end
HOME_OVERTRAVEL = 1_100_000
HOME_BACKOFF = 10_000   # 1 mm off the end stop
HOME_SPEED = 50_000   # counts/s for homing and returning to start

# ─── Validate & normalize configurations (no need change) ───
STREAMS = [s.strip().lower() for s in STREAMS]
STREAMS = list(dict.fromkeys(STREAMS))   # remove duplicates, keep order

_bad = [s for s in STREAMS if s not in STREAM_MAP]
if _bad:
    raise ValueError(f"Unknown stream(s): {_bad}. Valid options: radc, rfft, done")
if not STREAMS:
    raise ValueError("STREAMS is empty. Select at least one of: radc, rfft, done")
if not (0 <= RSET_CONFIG <= 8):
    raise ValueError("RSET_CONFIG must be 0-8")

SWEEP_COUNTS = int(round(SWEEP_LENGTH_MM * COUNTS_PER_MM))
if HOME_BACKOFF + SWEEP_COUNTS > STAGE_RANGE_COUNTS:
    raise ValueError("Backoff plus sweep length exceeds the stage travel")
if N_SWEEP_FRAMES < 2:
    raise ValueError("N_SWEEP_FRAMES must be at least 2")

# Session paths
SESSION_DIR = os.path.join(OUTPUT_BASE, SESSION_NAME)
BINS_DIR = os.path.join(SESSION_DIR, "bins")
CSV_PATH = os.path.join(SESSION_DIR, "scan_log.csv")

# Compute RDOT flag list from the selected streams
SELECTED_RDOT = [STREAM_MAP[s] for s in STREAMS]

# Summary
print(f"[cell 2] Session folder : {SESSION_DIR}")
print(f"[cell 2] Streams        : {STREAMS}  (RDOT = 0x{sum(r.value for r in SELECTED_RDOT):02X})")
print(f"[cell 2] RSET mode      : {RSET_CONFIG}")
print(f"[cell 2] Sweep          : {SWEEP_LENGTH_MM:.1f} mm, target {N_SWEEP_FRAMES} frames, sync on {sync_stream().upper()}")

## Cell 3 - Connect to the radar

Run once at the start of a session. This opens the connection, configures the radar with the RSET mode and selected streams, and creates the session folder.

Safe to re-run: if a connection is already open, it closes it first. Re-run this (then Cell 3C) after changing `RSET_CONFIG` or `STREAMS` in Cell 2.

The connection stays open until you run Cell 5 (or restart the kernel), so you **don't** need to re-run this between sweeps.

In [ ]:
# IF a connection is already open from a previous run, close it first so re-running this cell doesn't leave a dangling socket.
if 'vmd3' in globals() and vmd3 is not None:
    try:
        vmd3.sockTCP.settimeout(2.0)
        vmd3.disconnect()
        print("[cell 3] Closed existing connection.")
    except Exception as e:
        print(f"[cell 3] Warning closing old connection: {e}")
        try:
            vmd3.sockTCP.close()
        except Exception:
            pass
        try:
            vmd3.sockUDP.close()
        except Exception:
            pass
    vmd3 = None

# ─── Create the session folders + CSV (committed here, not in Cell 2) ───
os.makedirs(BINS_DIR, exist_ok=True)
if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "filename", "scan_pos_cm", "timestamp_start", "timestamp_stop",
            "duration_s", "n_frames", "rset_mode", "streams", "notes",
        ])
    print(f"[cell 3] Created {CSV_PATH}")
else:
    print(f"[cell 3] Appending to existing {CSV_PATH}")

# ─── Connect ────────────────────────────────────────────────────
print(f"[cell 3] Connecting to V-MD3 at {RADAR_IP}...")
vmd3 = VMD3(tcp_ip=RADAR_IP)
vmd3.connect()
vmd3.set_rset_config(RSET_CONFIG)
vmd3.set_output_config(SELECTED_RDOT)
vmd3.sockUDP.settimeout(UDP_TIMEOUT)
vmd3.sockUDP.setsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF, 8 * 1024 * 1024)   # buffers frames while Python queries the Galil

print(f"[cell 3] Ready. RSET={RSET_CONFIG}, streams={STREAMS}, "
      f"RDOT=0x{sum(r.value for r in SELECTED_RDOT):02X}")

## Cell 3B - Connect to the Galil and home the stage

Close GDK first. Run once per session. Drives into the start end stop, backs off 1 mm, and zeros the position there. Re-run if the stage is bumped or power cycled.

In [ ]:
# Close any previous Galil handle so re-running doesn't leave a stale connection
if 'galil' in globals() and galil is not None:
    try:
        galil.GClose()
    except Exception:
        pass

# Connect and stop anything left running from GDK
galil = gclib.py()
galil.GOpen(GALIL_ADDRESS)
print(f"[cell 3B] Connected: {galil.GInfo()}")
gcmd(f"ST{GALIL_AXIS}")
galil.GMotionComplete(GALIL_AXIS)
try:
    gcmd("HX")   # halt any program left running from GDK
except RuntimeError:
    pass

# Home to the start end
print("[cell 3B] Homing (up to ~20 s)...")
galil_home()
print(f"[cell 3B] Homed. Position {galil_state()[0]} counts = start of sweep.")

## Cell 3C - Measure the frame period

Times 30 radar cycles with the current streams and computes the sweep speed for `N_SWEEP_FRAMES` over `SWEEP_LENGTH_MM`. Re-run after changing STREAMS or RSET.

In [ ]:
N_TIMING_CYCLES = 30
WAVELENGTH_MM = 299_792_458 / 61.0e9 * 1e3


def measure_frame_period(n_cycles=N_TIMING_CYCLES):
    """Time n radar cycles and return the median cycle period in seconds."""
    header = HEADER_MAP[sync_stream()]
    drain_udp(vmd3)
    stamps = []
    while len(stamps) < n_cycles + 1:
        wait_for_frame(vmd3, header)
        stamps.append(time.perf_counter())
    return float(np.median(np.diff(stamps)))


# Measure and report the resulting sweep plan
MEASURED_FRAME_PERIOD_S = measure_frame_period()
speed = SWEEP_COUNTS / ((N_SWEEP_FRAMES - 1) * MEASURED_FRAME_PERIOD_S)
spacing_mm = SWEEP_LENGTH_MM / (N_SWEEP_FRAMES - 1)
print(f"[cell 3C] Frame period  : {MEASURED_FRAME_PERIOD_S*1e3:.1f} ms ({1/MEASURED_FRAME_PERIOD_S:.2f} Hz)")
print(f"[cell 3C] Sweep speed   : {speed:.0f} counts/s ({speed/COUNTS_PER_MM:.2f} mm/s)")
print(f"[cell 3C] Sweep time    : {SWEEP_COUNTS/speed:.1f} s for {N_SWEEP_FRAMES} frames")
print(f"[cell 3C] Frame spacing : {spacing_mm:.3f} mm (λ/4 = {WAVELENGTH_MM/4:.3f} mm)")

# Warn if the spacing is too coarse for alias-free SAR
if spacing_mm > WAVELENGTH_MM / 4:
    min_frames = int(np.ceil(SWEEP_LENGTH_MM / (WAVELENGTH_MM / 4))) + 1
    print(f"[cell 3C] Warning: spacing exceeds λ/4. Use N_SWEEP_FRAMES ≥ {min_frames} to avoid grating lobes.")

## Cell 4 - One continuous sweep

Returns the stage to start, waits for a radar cycle boundary, starts the sweep, and records until the stage stops and that cycle finishes. The speed is set so the sweep spans `N_SWEEP_FRAMES` frames (set in Cell 2, measured period from Cell 3C).

Saves `sweep_###.bin` plus `sweep_###_positions.csv` with one row per record in file order:

- `stream`, `stream_frame` - which frame in the `.bin` (e.g. RADC frame 12)
- `t_s` - arrival time since motion start
- `galil_counts` - raw encoder position
- `x_mm` - 0 at the start (right) end to `SWEEP_LENGTH_MM` at the far end
- `x_center_mm` - position from the sweep center, + at the right end, − at the left end
- `moving` - 1 if the stage was moving

Use `x_center_mm` (or `x_mm`) for SAR processing rather than assuming uniform spacing. Repeat for each sweep.

In [ ]:
SWEEP_NAME = "ball_sweep"   # saved as ball_sweep.bin, or ball_sweep_2.bin, _3... if the name is taken
SWEEP_NOTES = ""   # optional note for scan_log.csv


def next_sweep_path():
    """Path for SWEEP_NAME in BINS_DIR, adding _2, _3... instead of overwriting."""
    path = os.path.join(BINS_DIR, f"{SWEEP_NAME}.bin")
    i = 2
    while os.path.exists(path):
        path = os.path.join(BINS_DIR, f"{SWEEP_NAME}_{i}.bin")
        i += 1
    return path

def record_sweep(notes=""):
    """Run one continuous sweep and record every frame while the stage moves."""
    period = FRAME_PERIOD_OVERRIDE_S or globals().get("MEASURED_FRAME_PERIOD_S")
    if period is None:
        raise RuntimeError("No frame period. Run Cell 3C or set FRAME_PERIOD_OVERRIDE_S in Cell 2.")
    speed = int(round(SWEEP_COUNTS / ((N_SWEEP_FRAMES - 1) * period)))
    timeout_s = SWEEP_COUNTS / speed + 10
    half_mm = SWEEP_LENGTH_MM / 2

    # File paths and per-record bookkeeping
    filepath = next_sweep_path()
    filename = os.path.basename(filepath)
    pos_csv = filepath.replace(".bin", "_positions.csv")
    accepted = tuple(HEADER_MAP[s] for s in STREAMS)
    header_to_name = {HEADER_MAP[s]: s for s in STREAMS}
    sync_name = sync_stream()
    per_stream = {s: 0 for s in STREAMS}
    rows = []
    bytes_written = 0

    # Park at the start and load the sweep profile without starting it
    print("[cell 4] Moving to start...")
    galil_go_to_start()
    gcmd(f"SP{GALIL_AXIS}={speed}")
    gcmd(f"AC{GALIL_AXIS}={SWEEP_ACCEL}")
    gcmd(f"DC{GALIL_AXIS}={SWEEP_ACCEL}")
    gcmd(f"PR{GALIL_AXIS}={-START_END_SIGN * SWEEP_COUNTS}")

    # Wait for the end of a radar cycle so motion starts on a cycle boundary
    drain_udp(vmd3)
    wait_for_frame(vmd3, HEADER_MAP[sync_name])

    # Start the motion and the clock together
    gcmd(f"BG{GALIL_AXIS}")
    t0 = time.perf_counter()
    start_time = datetime.now()
    print(f"[cell 4] Sweeping {SWEEP_LENGTH_MM:.0f} mm at {speed/COUNTS_PER_MM:.2f} mm/s "
          f"(target {N_SWEEP_FRAMES} frames) → {filename}")

    # Record until the stage has stopped and the current cycle is complete
    out = open(filepath, "wb")
    try:
        while True:
            if time.perf_counter() - t0 > timeout_s:
                raise TimeoutError("Sweep took too long. Check the radar stream and the stage.")
            try:
                frame = read_any_frame(vmd3, accepted)
            except socket.timeout:
                continue
            t_frame = time.perf_counter() - t0
            pos, moving = galil_state()
            name = header_to_name[frame[0:4]]

            out.write(frame)
            bytes_written += len(frame)
            x_mm = -START_END_SIGN * pos / COUNTS_PER_MM
            x_center_mm = half_mm - x_mm
            rows.append([len(rows), name, per_stream[name], f"{t_frame:.4f}", pos,
                         f"{x_mm:.4f}", f"{x_center_mm:.4f}", int(moving)])
            per_stream[name] += 1

            if name == sync_name:
                print(f"   cycle {per_stream[name]:3d}   x = {x_mm:7.2f} mm   "
                      f"center {x_center_mm:+7.2f} mm   {'moving' if moving else 'stopped'}")
                if not moving:
                    break
    except BaseException:
        gcmd(f"ST{GALIL_AXIS}")
        raise
    finally:
        out.close()

    stop_time = datetime.now()
    duration = (stop_time - start_time).total_seconds()

    # Save the per-record position log
    with open(pos_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["record", "stream", "stream_frame", "t_s", "galil_counts",
                         "x_mm", "x_center_mm", "moving"])
        writer.writerows(rows)

    # Add a summary row to the session log
    with open(CSV_PATH, "a", newline="") as f:
        csv.writer(f).writerow([
            filename, "sweep",
            start_time.isoformat(timespec="seconds"),
            stop_time.isoformat(timespec="seconds"),
            f"{duration:.2f}", len(rows), RSET_CONFIG, "+".join(STREAMS),
            f"{notes} | {SWEEP_LENGTH_MM:g} mm at {speed} cts/s, target {N_SWEEP_FRAMES} frames",
        ])

    # Summarize the aperture using the first stream of each cycle
    first = [s for s in CYCLE_ORDER if s in STREAMS][0]
    xc = np.array([float(r[6]) for r in rows if r[1] == first])
    print(f"[cell 4] Saved {filename}: {len(rows)} records, {bytes_written/1e6:.2f} MB, {duration:.1f} s")
    print(f"[cell 4] {len(xc)} {first.upper()} frames (target {N_SWEEP_FRAMES}), "
          f"center position {xc[0]:+.2f} to {xc[-1]:+.2f} mm, mean spacing {np.mean(np.abs(np.diff(xc))):.3f} mm")
    print(f"[cell 4] Positions → {os.path.basename(pos_csv)}")


record_sweep(SWEEP_NOTES)

## Cell 5 - Disconnect

Run at the end of a session to stop the stage and close both the radar and Galil connections cleanly.

Safe to run even if nothing is connected since it just reports that there's nothing to close.

In [ ]:
if 'vmd3' in globals() and vmd3 is not None:
    try:
        vmd3.sockTCP.settimeout(2.0)
        vmd3.disconnect()
        print("[cell 5] Disconnected from V-MD3.")
    except Exception as e:
        print(f"[cell 5] Disconnect warning: {e}")
        # Force-close the sockets so the OS releases the ports
        try:
            vmd3.sockTCP.close()
        except Exception:
            pass
        try:
            vmd3.sockUDP.close()
        except Exception:
            pass
        print("[cell 5] Sockets force-closed.")
    finally:
        vmd3 = None
else:
    print("[cell 5] Nothing to disconnect (no open connection).")

# Stop the stage and close the Galil connection
if 'galil' in globals() and galil is not None:
    try:
        gcmd(f"ST{GALIL_AXIS}")
        galil.GClose()
        print("[cell 5] Disconnected from Galil.")
    except Exception as e:
        print(f"[cell 5] Galil close warning: {e}")
    finally:
        galil = None
else:
    print("[cell 5] No Galil connection to close.")

## Cell 6 - Convert .bin to long-format CSV

Decodes a saved `.bin` file into a lossless "long" CSV: one row per (frame, bin, chirp, channel) value, with complex I/Q split into separate `I` and `Q` columns.

Set `BIN_TO_CONVERT` to the sweep file you want, choose which streams to export, then run. The CSV is saved next to the `.bin` with `_long.csv` added.

Warning: long format is large. One RADC or RFFT frame is 32,768 rows, so a ~101-frame sweep with RADC + RFFT is about 6.6 million rows. Lossless but big.

In [ ]:
# ─── What to convert ───
BIN_TO_CONVERT = os.path.join(BINS_DIR, "sweep_001.bin")   # path to a .bin file
CONVERT_STREAMS = ["radc", "rfft"]   # which streams to export: radc and/or rfft
CONVERT_MODE = "2D"   # "2D" or "3D" (matches how it was recorded)

# ─── Where to save ───
OUTPUT_CSV_PATH = os.path.splitext(BIN_TO_CONVERT)[0] + "_long.csv"

# ─── Decoders ───
def _get_frames(filepath, header, expected_len):
    """Extract all valid payloads for a given header from a .bin file."""
    with open(filepath, "rb") as f:
        raw = f.read()
    frames, start = [], 0
    while True:
        idx = raw.find(header, start)
        if idx == -1:
            break
        if idx + 8 > len(raw):
            break
        payload_len = int.from_bytes(raw[idx+4:idx+8], byteorder="little")
        if payload_len == expected_len:
            p0, p1 = idx + 8, idx + 8 + expected_len
            if p1 <= len(raw):
                payload = raw[p0:p1]
                if header not in payload:   # reject header-in-payload false hits
                    frames.append(payload)
        start = idx + 1
    return frames


def _decode_radc_2d(frame):
    """Decode one 2D RADC payload to (samples, chirps, channels)."""
    raw = np.frombuffer(frame, dtype="<i2").reshape(64, 4, 256)
    q = raw[:, :, 0::2].astype(np.float64)
    i = raw[:, :, 1::2].astype(np.float64)
    complex_data = i + 1j * q
    return np.transpose(complex_data, (2, 0, 1))


def _decode_rfft_2d(frame):
    """Decode one 2D RFFT payload to (range_bin, chirp, ch)."""
    raw = np.frombuffer(frame, dtype="<i2").reshape(4, 128, 64, 2)
    i = raw[:, :, :, 1].astype(np.float64)
    q = raw[:, :, :, 0].astype(np.float64)
    complex_data = i + 1j * q
    return np.transpose(complex_data, (1, 2, 0))


# ─── Build the long dataframe ───
def bin_to_long_df(filepath, streams, mode):
    """Decode the chosen streams of a .bin file into one long dataframe."""
    expected_len = 131072 if mode == "2D" else 196608
    header_map = {"radc": b"RADC", "rfft": b"RFFT"}
    all_rows = []

    for s in streams:
        header = header_map[s]
        frames = _get_frames(filepath, header, expected_len)
        if not frames:
            print(f"  [{s}] no frames found, skipping")
            continue

        for f_idx, frame in enumerate(frames):
            cube = _decode_radc_2d(frame) if s == "radc" else _decode_rfft_2d(frame)

            # Flatten the (bin, chirp, channel) cube into index columns
            n_bin, n_chirp, n_chan = cube.shape
            bb, cc, hh = np.meshgrid(
                np.arange(n_bin), np.arange(n_chirp), np.arange(n_chan),
                indexing="ij"
            )
            df = pd.DataFrame({
                "file": os.path.basename(filepath),
                "stream": s,
                "frame": f_idx,
                "bin": bb.ravel(),   # sample index (radc) or range bin (rfft)
                "chirp": cc.ravel(),
                "channel": hh.ravel(),
                "I": np.real(cube).ravel(),
                "Q": np.imag(cube).ravel(),
            })
            all_rows.append(df)
            print(f"  [{s}] frame {f_idx}: {len(df)} rows")

    if not all_rows:
        print("No data converted.")
        return None
    return pd.concat(all_rows, ignore_index=True)


# ─── Run ───
print(f"[cell 6] Converting {os.path.basename(BIN_TO_CONVERT)} "
      f"(streams={CONVERT_STREAMS}, mode={CONVERT_MODE})...")

df = bin_to_long_df(BIN_TO_CONVERT, CONVERT_STREAMS, CONVERT_MODE)

if df is not None:
    # Write the CSV
    out_dir = os.path.dirname(OUTPUT_CSV_PATH)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"[cell 6] Wrote {OUTPUT_CSV_PATH}")
    print(f"[cell 6] Shape: {df.shape[0]} rows × {df.shape[1]} cols")

    # Per-stream row breakdown
    stream_counts = df["stream"].value_counts()
    for s in CONVERT_STREAMS:
        print(f"[cell 6]   {s}: {int(stream_counts.get(s, 0))} rows")
    print(f"[cell 6]   total: {len(df)} rows")
    print(df.head())


## Cell 7 — Quick background-subtraction plot

Run these final cells after recording a background scan followed by a measurement
scan. The processing follows the supplied dirt-comparison notebook: RADC decoding,
Hann-windowed range/Doppler FFTs, centered zero-Doppler extraction, coherent frame
averaging, per-RX complex background alignment, complex subtraction, and four-RX
range–azimuth beamforming.

If both filenames below are `None`, the newest two `.bin` files in `BINS_DIR` are
used in acquisition order: **older = background, newer = measurement**. For an
unambiguous record, enter both exact filenames.

The vertical axis is nominal range in meters using mode-0 spacing
`6 m / 128 = 0.046875 m/bin` (about 4.69 cm/bin). It does not include an empirical
range-offset calibration.


In [ ]:

# ─── Select the background and measurement ─────────────────────
# Leave BOTH as None to use the newest pair in BINS_DIR.
QUICK_BACKGROUND_FILENAME = "bg_bucket.bin"    # Example: "scan_0cm.bin"
QUICK_MEASUREMENT_FILENAME = "ball_0cm_1.bin"   # Example: "scan_0cm_1.bin"

# ─── Quick-plot display settings ────────────────────────────────
QUICK_PLOT_MAX_RANGE_M = 1.6        # None shows the full nominal 6 m
QUICK_DB_FLOOR = -35.0
QUICK_SAVE_FIGURE = False           # True saves quick_plot.png in SESSION_DIR

from pathlib import Path
import warnings
import matplotlib.pyplot as plt
from IPython.display import display

# V-MD3 RSET mode 0 / 2-D constants used above.
C_M_PER_S = 299_792_458.0
CARRIER_FREQUENCY_HZ = 61.0e9
RX_ELEMENT_SPACING_M = 2.464e-3
N_RANGE_BINS = 128
N_CHIRPS = 64
N_RX = 4
NOMINAL_MAXIMUM_RANGE_M = 6.0
RANGE_BIN_SPACING_M = NOMINAL_MAXIMUM_RANGE_M / N_RANGE_BINS

RANGE_WINDOW = "hann"
DOPPLER_WINDOW = "hann"
AZIMUTH_ANGLE_DEG = np.linspace(-70.0, 70.0, 281)
PLOT_MAX_RANGE_M = QUICK_PLOT_MAX_RANGE_M
DB_FLOOR = QUICK_DB_FLOOR
COLORMAP = "viridis"

# Same complex-alignment settings as the dirt-comparison notebook.
ALIGNMENT_RANGE_BIN_START = 1
ALIGNMENT_RANGE_BIN_STOP = 51
EXCLUDED_ALIGNMENT_BIN_RANGES = [(15, 25)]
MINIMUM_REFERENCE_LEVEL_DB = -35.0

print(f"Quick-plot binary folder: {Path(BINS_DIR).resolve()}")
print(f"Nominal range spacing: {RANGE_BIN_SPACING_M:.6f} m/bin")


### Quick-plot functions (run once after restarting the kernel)

In [ ]:

FRAME_HEADER_BYTES = 8
STREAM_CODE_TO_NAME = {b"RADC": "radc", b"RFFT": "rfft", b"DONE": "done"}
EXPECTED_PAYLOAD_BYTES = {"radc": 131_072, "rfft": 131_072, "done": 4}
RADC_FRAME_SHAPE = (N_RANGE_BINS, N_CHIRPS, N_RX)


def scan_vmd3_bin(path):
    """Return one manifest row per valid V-MD3 record."""
    path = Path(path)
    file_size = path.stat().st_size
    rows, offset, record_index = [], 0, 0
    stream_counts = {name: 0 for name in STREAM_CODE_TO_NAME.values()}

    with path.open("rb") as f:
        while offset < file_size:
            f.seek(offset)
            header = f.read(FRAME_HEADER_BYTES)
            if len(header) != FRAME_HEADER_BYTES:
                raise EOFError(f"{path.name}: incomplete header at byte {offset:,}")

            stream_code = header[:4]
            if stream_code not in STREAM_CODE_TO_NAME:
                raise ValueError(
                    f"{path.name}: unknown stream code {stream_code!r} "
                    f"at byte {offset:,}"
                )
            stream = STREAM_CODE_TO_NAME[stream_code]
            payload_length = int.from_bytes(header[4:8], "little", signed=False)
            expected = EXPECTED_PAYLOAD_BYTES[stream]
            if payload_length != expected:
                raise ValueError(
                    f"{path.name}: {stream.upper()} payload is "
                    f"{payload_length:,} bytes; expected {expected:,}"
                )

            payload_offset = offset + FRAME_HEADER_BYTES
            next_offset = payload_offset + payload_length
            if next_offset > file_size:
                raise EOFError(f"{path.name}: record {record_index} extends past EOF")

            rows.append({
                "record_index": record_index,
                "stream": stream,
                "stream_frame_index": stream_counts[stream],
                "payload_offset": payload_offset,
                "payload_length": payload_length,
            })
            stream_counts[stream] += 1
            record_index += 1
            offset = next_offset

    return pd.DataFrame(rows)


def decode_radc_2d(payload):
    """Decode one mode-0 RADC payload to (sample, chirp, RX)."""
    expected = EXPECTED_PAYLOAD_BYTES["radc"]
    if len(payload) != expected:
        raise ValueError(f"Expected {expected:,} bytes; received {len(payload):,}")

    raw = np.frombuffer(payload, dtype="<i2").reshape(N_CHIRPS, N_RX, 2*N_RANGE_BINS)
    q = raw[:, :, 0::2].astype(np.float64)
    i = raw[:, :, 1::2].astype(np.float64)
    cube = np.transpose(i + 1j*q, (2, 0, 1))
    if cube.shape != RADC_FRAME_SHAPE:
        raise ValueError(f"Decoded shape {cube.shape}; expected {RADC_FRAME_SHAPE}")
    return cube


def load_radc_stack(path):
    """Load all RADC records as (frame, sample, chirp, RX)."""
    path = Path(path)
    manifest = scan_vmd3_bin(path)
    records = manifest.loc[manifest["stream"] == "radc"]
    if records.empty:
        raise ValueError(f"No RADC records found in {path.name}")

    cubes = []
    with path.open("rb") as f:
        for record in records.itertuples(index=False):
            f.seek(int(record.payload_offset))
            payload = f.read(int(record.payload_length))
            if len(payload) != int(record.payload_length):
                raise EOFError(f"Could not read complete payload from {path.name}")
            cubes.append(decode_radc_2d(payload))
    return np.stack(cubes, axis=0), manifest



WINDOW_FUNCTIONS = {
    "rectangular": lambda length: np.ones(length),
    "hann": np.hanning,
    "hamming": np.hamming,
    "blackman": np.blackman,
}


def create_window(name, length):
    key = name.lower()
    if key not in WINDOW_FUNCTIONS:
        raise ValueError(f"Unsupported window {name!r}")
    return WINDOW_FUNCTIONS[key](length)


def process_radc_stack_to_zero_doppler(radc_stack, range_window=RANGE_WINDOW,
                                        doppler_window=DOPPLER_WINDOW):
    """RADC stack -> coherent frame mean in (range bin, RX) space."""
    x = np.asarray(radc_stack, dtype=np.complex128).copy()
    if x.ndim != 4 or x.shape[1:] != RADC_FRAME_SHAPE:
        raise ValueError(
            f"Expected (frame, {N_RANGE_BINS}, {N_CHIRPS}, {N_RX}); got {x.shape}"
        )

    range_w = create_window(range_window, x.shape[1])
    doppler_w = create_window(doppler_window, x.shape[2])
    range_fft = np.fft.fft(x * range_w[None, :, None, None], axis=1)
    range_doppler = np.fft.fft(
        range_fft * doppler_w[None, None, :, None], axis=2
    )
    range_doppler = np.fft.fftshift(range_doppler, axes=2)
    zero_bin = range_doppler.shape[2] // 2
    zero_stack = range_doppler[:, :, zero_bin, :]
    return np.mean(zero_stack, axis=0), zero_stack, zero_bin


def make_alignment_mask(n_range_bins):
    mask = np.zeros(n_range_bins, dtype=bool)
    mask[ALIGNMENT_RANGE_BIN_START:ALIGNMENT_RANGE_BIN_STOP] = True
    for start_bin, stop_bin in EXCLUDED_ALIGNMENT_BIN_RANGES:
        mask[start_bin:stop_bin + 1] = False
    return mask


def align_complex_background(measurement_zero, background_zero):
    """Align background to measurement with one complex coefficient per RX."""
    measurement_zero = np.asarray(measurement_zero)
    background_zero = np.asarray(background_zero)
    if measurement_zero.shape != background_zero.shape:
        raise ValueError(
            f"Shape mismatch: measurement {measurement_zero.shape}, "
            f"background {background_zero.shape}"
        )

    base_mask = make_alignment_mask(measurement_zero.shape[0])
    min_level = 10**(MINIMUM_REFERENCE_LEVEL_DB / 20)
    tiny = np.finfo(float).tiny
    meas_norm = np.abs(measurement_zero) / max(float(np.abs(measurement_zero).max()), tiny)
    bg_norm = np.abs(background_zero) / max(float(np.abs(background_zero).max()), tiny)
    coefficients = np.zeros(measurement_zero.shape[1], dtype=np.complex128)
    rows = []

    for rx in range(measurement_zero.shape[1]):
        channel_mask = (
            base_mask
            & (meas_norm[:, rx] >= min_level)
            & (bg_norm[:, rx] >= min_level)
        )
        meas_ref = measurement_zero[channel_mask, rx]
        bg_ref = background_zero[channel_mask, rx]
        if meas_ref.size == 0:
            raise ValueError(
                f"RX {rx}: no valid alignment bins. Adjust the alignment region, "
                "excluded target bins, or minimum reference level."
            )

        alpha = np.sum(meas_ref*np.conj(bg_ref)) / max(
            float(np.sum(np.abs(bg_ref)**2)), tiny
        )
        coefficients[rx] = alpha
        corr = np.abs(np.sum(meas_ref*np.conj(bg_ref))) / max(
            float(np.sqrt(np.sum(np.abs(meas_ref)**2)*np.sum(np.abs(bg_ref)**2))),
            tiny,
        )
        rows.append({
            "RX": rx,
            "reference bins used": int(meas_ref.size),
            "|alpha|": float(np.abs(alpha)),
            "alpha phase (deg)": float(np.rad2deg(np.angle(alpha))),
            "complex correlation": float(corr),
        })

    aligned_background = background_zero * coefficients[None, :]
    residual = measurement_zero - aligned_background
    return aligned_background, residual, coefficients, pd.DataFrame(rows)


def conventional_range_azimuth(range_rx_data, azimuth_angle_deg=AZIMUTH_ANGLE_DEG):
    """Four-RX conventional beamforming: (range, RX) -> (range, azimuth)."""
    n_rx = range_rx_data.shape[1]
    rx_positions_m = (np.arange(n_rx) - (n_rx - 1)/2) * RX_ELEMENT_SPACING_M
    wavelength_m = C_M_PER_S / CARRIER_FREQUENCY_HZ
    k = 2*np.pi / wavelength_m
    theta = np.deg2rad(azimuth_angle_deg)
    steering = np.exp(-1j*k*rx_positions_m[:, None]*np.sin(theta)[None, :])
    return range_rx_data @ np.conj(steering)


def resolve_scan_pair(data_dir, background_filename=None, measurement_filename=None):
    """Resolve explicit names, or select the newest two files in acquisition order."""
    bins_dir = Path(data_dir) / "bins"
    if not bins_dir.is_dir():
        raise FileNotFoundError(f"Binary-data folder not found: {bins_dir}")

    if (background_filename is None) != (measurement_filename is None):
        raise ValueError("Set both filenames, or leave both as None")

    if background_filename is not None:
        background_file = bins_dir / background_filename
        measurement_file = bins_dir / measurement_filename
    else:
        candidates = sorted(bins_dir.glob("*.bin"), key=lambda p: p.stat().st_mtime)
        if len(candidates) < 2:
            raise FileNotFoundError(
                f"Need at least two .bin files in {bins_dir}; found {len(candidates)}"
            )
        background_file, measurement_file = candidates[-2:]
        warnings.warn(
            "Auto-selected the newest pair by file time. Confirm that the older file "
            "is the background and the newer file is the measurement."
        )

    for path in (background_file, measurement_file):
        if not path.is_file():
            raise FileNotFoundError(path)
    if background_file.resolve() == measurement_file.resolve():
        raise ValueError("Background and measurement must be different files")
    return background_file, measurement_file


def common_reference_db(complex_map, reference, floor_db=DB_FLOOR):
    tiny = np.finfo(float).tiny
    db = 20*np.log10(np.maximum(np.abs(complex_map), tiny) / max(float(reference), tiny))
    return np.maximum(db, floor_db)


def quick_plot_vmd3(data_dir, background_filename=None, measurement_filename=None,
                    plot_max_range_m=PLOT_MAX_RANGE_M, floor_db=DB_FLOOR,
                    figsize=(19, 6.5), save_figure=None):
    """
    Process one background/measurement pair and make the three-panel lab plot.

    Returns a dictionary containing the selected files, complex maps, dB maps,
    alignment diagnostics, peak estimate, and Matplotlib figure/axes.
    """
    background_file, measurement_file = resolve_scan_pair(
        data_dir, background_filename, measurement_filename
    )
    print(f"Background : {background_file.name}")
    print(f"Measurement: {measurement_file.name}")

    background_radc, background_manifest = load_radc_stack(background_file)
    measurement_radc, measurement_manifest = load_radc_stack(measurement_file)
    if background_radc.shape[1:] != measurement_radc.shape[1:]:
        raise ValueError(
            f"RADC cube mismatch: background {background_radc.shape}, "
            f"measurement {measurement_radc.shape}"
        )

    background_zero, background_zero_stack, bg_zero_bin = (
        process_radc_stack_to_zero_doppler(background_radc)
    )
    measurement_zero, measurement_zero_stack, meas_zero_bin = (
        process_radc_stack_to_zero_doppler(measurement_radc)
    )
    if bg_zero_bin != meas_zero_bin:
        raise RuntimeError("Background and measurement zero-Doppler bins differ")

    aligned_background, residual, coefficients, alignment_table = (
        align_complex_background(measurement_zero, background_zero)
    )
    measurement_map = conventional_range_azimuth(measurement_zero)
    background_map = conventional_range_azimuth(aligned_background)
    residual_map = conventional_range_azimuth(residual)

    common_reference = max(float(np.abs(measurement_map).max()), np.finfo(float).tiny)
    db_maps = [
        common_reference_db(measurement_map, common_reference, floor_db),
        common_reference_db(background_map, common_reference, floor_db),
        common_reference_db(residual_map, common_reference, floor_db),
    ]

    n_range_bins = residual_map.shape[0]
    range_axis_m = np.arange(n_range_bins)*RANGE_BIN_SPACING_M
    if plot_max_range_m is None:
        plot_stop = n_range_bins
    else:
        plot_stop = min(n_range_bins, int(np.floor(plot_max_range_m/RANGE_BIN_SPACING_M)) + 1)
        plot_stop = max(1, plot_stop)

    # Use bin-edge coordinates so each image row is centered on its nominal range.
    y_bottom = -0.5*RANGE_BIN_SPACING_M
    y_top = (plot_stop - 0.5)*RANGE_BIN_SPACING_M
    titles = [
        "Raw measurement\n(no subtraction)",
        "Aligned background scan",
        "Measurement − aligned background",
    ]
    fig, axes = plt.subplots(
        1, 3, figsize=figsize, sharex=True, sharey=True, constrained_layout=True
    )
    for ax, title, data in zip(axes, titles, db_maps):
        im = ax.imshow(
            data[:plot_stop, :], origin="lower", aspect="auto",
            extent=[AZIMUTH_ANGLE_DEG[0], AZIMUTH_ANGLE_DEG[-1], y_bottom, y_top],
            vmin=floor_db, vmax=0.0, cmap=COLORMAP,
        )
        ax.axvline(0.0, color="white", linestyle="--", linewidth=1.0, alpha=0.8)
        ax.set_title(title)
        ax.set_xlabel("Azimuth angle (degrees, provisional sign)")
        ax.grid(False)
    axes[0].set_ylabel("Nominal range (m; uncalibrated)")
    axes[0].set_ylim(0.0, max(range_axis_m[plot_stop - 1], RANGE_BIN_SPACING_M))
    colorbar = fig.colorbar(im, ax=axes, shrink=0.88, pad=0.02)
    colorbar.set_label("Magnitude relative to raw measurement maximum (dB)")
    fig.suptitle("V-MD3 Lab Quick Plot — Complex Background Subtraction", fontsize=15)

    search = np.abs(residual_map[:plot_stop, :])
    peak_range_bin, peak_angle_index = np.unravel_index(np.argmax(search), search.shape)
    peak = {
        "range bin": int(peak_range_bin),
        "nominal range (m)": float(range_axis_m[peak_range_bin]),
        "azimuth (deg, provisional sign)": float(AZIMUTH_ANGLE_DEG[peak_angle_index]),
        "residual level relative to raw maximum (dB)": float(db_maps[2][peak_range_bin, peak_angle_index]),
    }

    print(f"RADC frames: background={background_radc.shape[0]}, measurement={measurement_radc.shape[0]}")
    print(f"Centered zero-Doppler bin: {meas_zero_bin}")
    display(alignment_table)
    display(pd.DataFrame([peak]))

    if save_figure is not None:
        save_path = Path(save_figure)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print(f"Saved figure: {save_path}")

    return {
        "background_file": background_file,
        "measurement_file": measurement_file,
        "background_manifest": background_manifest,
        "measurement_manifest": measurement_manifest,
        "measurement_range_azimuth": measurement_map,
        "aligned_background_range_azimuth": background_map,
        "residual_range_azimuth": residual_map,
        "measurement_db": db_maps[0],
        "aligned_background_db": db_maps[1],
        "residual_db": db_maps[2],
        "range_axis_m": range_axis_m,
        "azimuth_axis_deg": AZIMUTH_ANGLE_DEG,
        "alignment_coefficients": coefficients,
        "alignment_table": alignment_table,
        "peak": peak,
        "figure": fig,
        "axes": axes,
    }


### Generate the three-panel lab plot

In [ ]:

quick_results = quick_plot_vmd3(
    Path(SESSION_DIR),
    background_filename=QUICK_BACKGROUND_FILENAME,
    measurement_filename=QUICK_MEASUREMENT_FILENAME,
    plot_max_range_m=QUICK_PLOT_MAX_RANGE_M,
    floor_db=QUICK_DB_FLOOR,
    save_figure=(Path(SESSION_DIR) / "quick_plot.png") if QUICK_SAVE_FIGURE else None,
)
plt.show()


In [ ]:
print(Path(BINS_DIR).resolve())# Target location
EXPECTED_SLANT_RANGE_M = 0.80
EXPECTED_GROUND_RANGE_M = 0.62
SEARCH_HALF_WIDTH_M = 0.20   # Search from 0.60–1.00 m slant range

# Retrieve the processed maps
measurement_map = quick_results["measurement_range_azimuth"]
background_map = quick_results["aligned_background_range_azimuth"]
residual_map = quick_results["residual_range_azimuth"]

range_axis_m = quick_results["range_axis_m"]
azimuth_axis_deg = quick_results["azimuth_axis_deg"]

# Use the raw measurement maximum as the common dB reference
common_reference = max(
    float(np.abs(measurement_map).max()),
    np.finfo(float).tiny,
)

def to_common_db(data):
    db = 20 * np.log10(
        np.maximum(np.abs(data), np.finfo(float).tiny)
        / common_reference
    )
    return np.maximum(db, QUICK_DB_FLOOR)

db_maps = [
    to_common_db(measurement_map),
    to_common_db(background_map),
    to_common_db(residual_map),
]

# Determine displayed range
if QUICK_PLOT_MAX_RANGE_M is None:
    plot_stop = len(range_axis_m)
else:
    plot_stop = min(
        len(range_axis_m),
        int(np.floor(QUICK_PLOT_MAX_RANGE_M / RANGE_BIN_SPACING_M)) + 1,
    )

# Search for the brightest residual near the expected target range
search_min_m = EXPECTED_SLANT_RANGE_M - SEARCH_HALF_WIDTH_M
search_max_m = EXPECTED_SLANT_RANGE_M + SEARCH_HALF_WIDTH_M

search_bins = np.where(
    (range_axis_m >= search_min_m)
    & (range_axis_m <= search_max_m)
    & (np.arange(len(range_axis_m)) < plot_stop)
)[0]

search_map = np.abs(residual_map[search_bins, :])

local_range_index, peak_angle_index = np.unravel_index(
    np.argmax(search_map),
    search_map.shape,
)

peak_range_bin = search_bins[local_range_index]
peak_range_m = range_axis_m[peak_range_bin]
peak_angle_deg = azimuth_axis_deg[peak_angle_index]
peak_db = db_maps[2][peak_range_bin, peak_angle_index]

print(f"Brightest gated residual:")
print(f"  Range bin: {peak_range_bin}")
print(f"  Nominal slant range: {peak_range_m:.3f} m")
print(f"  Azimuth: {peak_angle_deg:.1f}°")
print(f"  Residual level: {peak_db:.1f} dB")

# Plot the three maps with the detected target marked
titles = [
    "Raw measurement\n(no subtraction)",
    "Aligned background scan",
    "Measurement − aligned background",
]

fig, axes = plt.subplots(
    1, 3,
    figsize=(19, 6.5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

y_top = (plot_stop - 0.5) * RANGE_BIN_SPACING_M

for ax, title, data in zip(axes, titles, db_maps):
    im = ax.imshow(
        data[:plot_stop, :],
        origin="lower",
        aspect="auto",
        extent=[
            azimuth_axis_deg[0],
            azimuth_axis_deg[-1],
            -0.5 * RANGE_BIN_SPACING_M,
            y_top,
        ],
        vmin=QUICK_DB_FLOOR,
        vmax=0,
        cmap="viridis",
    )

    ax.axvline(
        0,
        color="white",
        linestyle="--",
        linewidth=1,
        alpha=0.8,
    )

    # Expected slant range
    ax.axhline(
        EXPECTED_SLANT_RANGE_M,
        color="white",
        linestyle=":",
        linewidth=1.2,
        alpha=0.9,
    )

    # Brightest residual within the search gate
    ax.plot(
        peak_angle_deg,
        peak_range_m,
        marker="x",
        color="red",
        markersize=12,
        markeredgewidth=2.5,
        linestyle="none",
    )

    ax.set_title(title)
    ax.set_xlabel("Azimuth angle (degrees, provisional sign)")

axes[0].set_ylabel("Nominal slant range (m; uncalibrated)")
axes[0].set_ylim(0, range_axis_m[plot_stop - 1])

axes[2].annotate(
    f"Peak: {peak_range_m:.2f} m, {peak_angle_deg:.1f}°\n"
    f"{peak_db:.1f} dB",
    xy=(peak_angle_deg, peak_range_m),
    xytext=(12, 12),
    textcoords="offset points",
    color="white",
    bbox=dict(facecolor="black", alpha=0.7, edgecolor="none"),
)

colorbar = fig.colorbar(im, ax=axes, shrink=0.88, pad=0.02)
colorbar.set_label("Magnitude relative to raw measurement maximum (dB)")

fig.suptitle(
    "V-MD3 Quick Plot — "
    f"Expected target: {EXPECTED_SLANT_RANGE_M:.2f} m slant / "
    f"{EXPECTED_GROUND_RANGE_M:.2f} m ground",
    fontsize=15,
)

plt.show()
print([p.name for p in Path(BINS_DIR).glob("*.bin")])

In [ ]:
# ============================================================
# Three-panel quick plot with target markers and amplitudes
# ============================================================

# Expected target position
EXPECTED_SLANT_RANGE_M = 0.80
EXPECTED_GROUND_RANGE_M = 0.62

# Search for the brightest residual from 0.60–1.00 m
SEARCH_HALF_WIDTH_M = 0.20

# ------------------------------------------------------------
# Retrieve processed range–azimuth maps
# ------------------------------------------------------------

measurement_map = quick_results["measurement_range_azimuth"]
background_map = quick_results["aligned_background_range_azimuth"]
residual_map = quick_results["residual_range_azimuth"]

range_axis_m = quick_results["range_axis_m"]
azimuth_axis_deg = quick_results["azimuth_axis_deg"]

# ------------------------------------------------------------
# Convert all maps to one common dB reference
# ------------------------------------------------------------

common_reference = max(
    float(np.abs(measurement_map).max()),
    np.finfo(float).tiny,
)

def to_common_db(data):
    magnitude = np.maximum(
        np.abs(data),
        np.finfo(float).tiny,
    )

    db = 20 * np.log10(
        magnitude / common_reference
    )

    return np.maximum(db, QUICK_DB_FLOOR)

measurement_db = to_common_db(measurement_map)
background_db = to_common_db(background_map)
residual_db = to_common_db(residual_map)

db_maps = [
    measurement_db,
    background_db,
    residual_db,
]

# ------------------------------------------------------------
# Determine displayed range
# ------------------------------------------------------------

if QUICK_PLOT_MAX_RANGE_M is None:
    plot_stop = len(range_axis_m)
else:
    plot_stop = min(
        len(range_axis_m),
        int(
            np.floor(
                QUICK_PLOT_MAX_RANGE_M
                / RANGE_BIN_SPACING_M
            )
        ) + 1,
    )

# ------------------------------------------------------------
# Search for the brightest residual near the target range
# ------------------------------------------------------------

search_min_m = max(
    0,
    EXPECTED_SLANT_RANGE_M
    - SEARCH_HALF_WIDTH_M,
)

search_max_m = (
    EXPECTED_SLANT_RANGE_M
    + SEARCH_HALF_WIDTH_M
)

search_bins = np.where(
    (range_axis_m >= search_min_m)
    & (range_axis_m <= search_max_m)
    & (np.arange(len(range_axis_m)) < plot_stop)
)[0]

if search_bins.size == 0:
    raise ValueError(
        "The target search range does not overlap "
        "the displayed range."
    )

search_map = np.abs(
    residual_map[search_bins, :]
)

local_range_index, peak_angle_index = (
    np.unravel_index(
        np.argmax(search_map),
        search_map.shape,
    )
)

peak_range_bin = int(
    search_bins[local_range_index]
)

peak_range_m = float(
    range_axis_m[peak_range_bin]
)

peak_angle_deg = float(
    azimuth_axis_deg[peak_angle_index]
)

# ------------------------------------------------------------
# Read amplitudes at the detected target location
# ------------------------------------------------------------

measurement_at_target_db = float(
    measurement_db[
        peak_range_bin,
        peak_angle_index,
    ]
)

background_at_target_db = float(
    background_db[
        peak_range_bin,
        peak_angle_index,
    ]
)

residual_at_target_db = float(
    residual_db[
        peak_range_bin,
        peak_angle_index,
    ]
)

print("Brightest residual inside target search gate")
print(
    f"  Search range: "
    f"{search_min_m:.2f}–{search_max_m:.2f} m"
)
print(f"  Range bin: {peak_range_bin}")
print(
    f"  Nominal slant range: "
    f"{peak_range_m:.3f} m"
)
print(
    f"  Azimuth: "
    f"{peak_angle_deg:.1f}°"
)
print(
    f"  Raw measurement: "
    f"{measurement_at_target_db:.1f} dB"
)
print(
    f"  Aligned background: "
    f"{background_at_target_db:.1f} dB"
)
print(
    f"  Background-subtracted residual: "
    f"{residual_at_target_db:.1f} dB"
)

# ------------------------------------------------------------
# Plot the three range–azimuth maps
# ------------------------------------------------------------

titles = [
    "Raw Measurement\n(No Subtraction)",
    "Aligned Background Scan",
    "Measurement − Aligned Background",
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(19, 6.5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

y_bottom = -0.5 * RANGE_BIN_SPACING_M

y_top = (
    plot_stop - 0.5
) * RANGE_BIN_SPACING_M

for ax, title, data in zip(
    axes,
    titles,
    db_maps,
):
    im = ax.imshow(
        data[:plot_stop, :],
        origin="lower",
        aspect="auto",
        extent=[
            azimuth_axis_deg[0],
            azimuth_axis_deg[-1],
            y_bottom,
            y_top,
        ],
        vmin=QUICK_DB_FLOOR,
        vmax=0,
        cmap="viridis",
    )

    # Boresight
    ax.axvline(
        0,
        color="white",
        linestyle="--",
        linewidth=1,
        alpha=0.8,
    )

    # Expected slant range
    ax.axhline(
        EXPECTED_SLANT_RANGE_M,
        color="white",
        linestyle=":",
        linewidth=1.2,
        alpha=0.9,
    )

    # Detected target location
    ax.plot(
        peak_angle_deg,
        peak_range_m,
        marker="x",
        color="red",
        markersize=12,
        markeredgewidth=2.5,
        linestyle="none",
    )

    ax.set_title(title)
    ax.set_xlabel(
        "Azimuth angle "
        "(degrees, provisional sign)"
    )
    ax.grid(False)

axes[0].set_ylabel(
    "Nominal slant range "
    "(m; uncalibrated)"
)

axes[0].set_ylim(
    0,
    range_axis_m[plot_stop - 1],
)

# ------------------------------------------------------------
# Amplitude labels
# ------------------------------------------------------------

axes[0].annotate(
    f"Measurement: "
    f"{measurement_at_target_db:.1f} dB",
    xy=(
        peak_angle_deg,
        peak_range_m,
    ),
    xytext=(12, 12),
    textcoords="offset points",
    color="white",
    fontsize=10,
    bbox=dict(
        facecolor="black",
        alpha=0.7,
        edgecolor="none",
        pad=3,
    ),
)

axes[1].annotate(
    f"Aligned BG: "
    f"{background_at_target_db:.1f} dB",
    xy=(
        peak_angle_deg,
        peak_range_m,
    ),
    xytext=(12, 12),
    textcoords="offset points",
    color="white",
    fontsize=10,
    bbox=dict(
        facecolor="black",
        alpha=0.7,
        edgecolor="none",
        pad=3,
    ),
)

axes[2].annotate(
    f"Residual: "
    f"{residual_at_target_db:.1f} dB\n"
    f"{peak_range_m:.2f} m, "
    f"{peak_angle_deg:.1f}°",
    xy=(
        peak_angle_deg,
        peak_range_m,
    ),
    xytext=(12, 12),
    textcoords="offset points",
    color="white",
    fontsize=10,
    bbox=dict(
        facecolor="black",
        alpha=0.7,
        edgecolor="none",
        pad=3,
    ),
)

# ------------------------------------------------------------
# Shared colorbar and title
# ------------------------------------------------------------

colorbar = fig.colorbar(
    im,
    ax=axes,
    shrink=0.88,
    pad=0.02,
)

colorbar.set_label(
    "Magnitude relative to raw "
    "measurement maximum (dB)"
)

fig.suptitle(
    "V-MD3 Quick Plot — "
    f"Expected Target: "
    f"{EXPECTED_SLANT_RANGE_M:.2f} m Slant / "
    f"{EXPECTED_GROUND_RANGE_M:.2f} m Ground",
    fontsize=15,
)

plt.show()